# CampusVoice-RT: 재현 노트북 (Colab, NVIDIA L4)

런타임 → GPU 유형을 **L4**로 설정한 뒤 위에서부터 실행. 모든 베이스라인(B1~B4)을 같은 런타임·같은 VAD 설정으로 측정한다.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/ekgP525/asr-challenge.git
%cd asr-challenge
!pip install -q -U "qwen-asr[vllm]" silero-vad soundfile librosa matplotlib
!python -c "from importlib.metadata import version as v; print('torch', v('torch'), 'vllm', v('vllm'), 'qwen-asr', v('qwen-asr'))"

## Hugging Face 인증

Colab 공용 IP는 익명 요청이 속도 제한(429)에 걸린다. 읽기 전용 토큰을 Colab 보안 비밀(🔑)에 `HF_TOKEN`으로 넣고 노트북 접근을 허용한 뒤 실행.

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!huggingface-cli whoami

## 평가셋 다운로드 (Hugging Face Datasets, 링크 TBD)

In [ ]:
!pip install -q huggingface_hub
!huggingface-cli download ekgP115/campusvoice-rt-eval --repo-type dataset --local-dir data/eval
!ls data/eval/audio && wc -l data/eval/manifest.jsonl

## 베이스라인 실행 → 채점

시뮬레이터는 가상 시계를 쓰므로 실행 시간은 모델 연산 시간뿐이다(오프라인 수 분, 스트리밍은 청크마다 전체 오디오를 재디코드해 더 길다). 세션이 끊기면 `--conditions clean,snr20` 처럼 조건을 나눠 돌리고 `runs/<sid>/`를 Drive에 보관한다. 빠른 점검은 `--limit 20 --repeats 1`.

In [ ]:
import subprocess
# each system in its own process so GPU memory is released between models
for sid in ['B1','B2','B3','B4']:
    subprocess.run(['python','baseline/run_qwen3_asr.py','--config',f'baseline/configs/{sid}.json',
                    '--manifest','data/eval/manifest.jsonl','--audio-root','data/eval','--out-dir',f'runs/{sid}'], check=True)
    subprocess.run(['python','evaluation/evaluate.py','--config',f'baseline/configs/{sid}.json',
                    '--manifest','data/eval/manifest.jsonl','--predictions',f'runs/{sid}/predictions.jsonl',
                    '--latency',f'runs/{sid}/latency.jsonl','--env',f'runs/{sid}/environment.json',
                    '--hardware','NVIDIA L4'], check=True)

## 리더보드

In [ ]:
!python evaluation/make_leaderboard.py
from IPython.display import Image, Markdown, display
display(Markdown(open('results/leaderboard.md').read()))
display(Image('results/latency_cer_paths.png'))